# 04. Modeling and Evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
df_train = pd.read_csv("../data/train_engineered.csv")

In [ ]:
engines = df_train["engine_id"].unique()
np.random.seed(42)
train_engines = np.random.choice(engines, size=int(0.8 * len(engines)), replace=False)
train_set = df_train[df_train["engine_id"].isin(train_engines)]
val_set = df_train[~df_train["engine_id"].isin(train_engines)]
features = [c for c in train_set.columns if c not in ["engine_id", "cycle", "RUL"]]
target = "RUL"
X_train, y_train = train_set[features], train_set[target]
X_val, y_val = val_set[features], val_set[target]

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)
rmse_lr = np.sqrt(mean_squared_error(y_val, lr.predict(X_val)))
print(f"Linear Regression RMSE: {rmse_lr:.2f}")

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rmse_rf = np.sqrt(mean_squared_error(y_val, rf.predict(X_val)))
print(f"Random Forest RMSE: {rmse_rf:.2f}")

In [ ]:
xgb = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
preds_xgb = xgb.predict(X_val)
rmse_xgb = np.sqrt(mean_squared_error(y_val, preds_xgb))
print(f"XGBoost RMSE: {rmse_xgb:.2f}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(y_val, preds_xgb, alpha=0.3, color="blue")
plt.plot([0, y_val.max()], [0, y_val.max()], "r--")
plt.xlabel("Actual RUL")
plt.ylabel("Predicted RUL")
plt.title("Actual vs Predicted RUL (XGBoost)")
plt.savefig("../figures/actual_vs_predicted.png", bbox_inches="tight")
plt.show()

In [ ]:
importance = pd.DataFrame({"Feature": features, "Importance": rf.feature_importances_}).sort_values("Importance", ascending=False)
plt.figure(figsize=(10, 8))
sns.barplot(x="Importance", y="Feature", data=importance.head(20), palette="viridis")
plt.title("Top 20 Features Importance (Random Forest)")
plt.savefig("../figures/feature_importance.png", bbox_inches="tight")
plt.show()